# Threat Intelligence Aggregator

**Author:** Snigdha Aswal
**Program:** IBM SkillsBuild Cyber Security Bootcamp 2026 (BharatCares)
**Project topic:** #12 - Threat Intelligence Aggregator

This notebook takes an **indicator of compromise (IoC)** - an IP address, a domain, or a file hash - queries two live threat-intelligence sources (**VirusTotal** and **AlienVault OTX**), combines the results into a single **risk score**, and gives a verdict (Low risk / Suspicious / Malicious).

**Pipeline:** input -> detect type -> query VirusTotal -> query OTX -> combine + score -> report -> save results

## 1. Setup

In [ ]:
import os
import re
import json
import time
import ipaddress
from datetime import datetime, timezone

import requests
import pandas as pd
import matplotlib.pyplot as plt

try:
    from dotenv import load_dotenv
    load_dotenv()          # reads VT_API_KEY and OTX_API_KEY from a local .env file
except ImportError:
    pass

print("Libraries loaded.")

## 2. API keys

Keys are read from environment variables (or a `.env` file). **Never hard-code keys or push them to GitHub.**

Create a `.env` file next to this notebook:
```
VT_API_KEY=your_virustotal_key
OTX_API_KEY=your_alienvault_otx_key
```
If the variables are missing, you will be asked to type them in.

In [ ]:
from getpass import getpass

VT_API_KEY = os.getenv("VT_API_KEY") or getpass("Enter your VirusTotal API key: ")
OTX_API_KEY = os.getenv("OTX_API_KEY") or getpass("Enter your AlienVault OTX API key: ")

VT_BASE = "https://www.virustotal.com/api/v3"
OTX_BASE = "https://otx.alienvault.com/api/v1/indicators"
TIMEOUT = 20  # seconds per request

## 3. Detect the indicator type

The tool accepts one input and works out whether it is an IPv4/IPv6 address, a file hash (MD5 / SHA-1 / SHA-256) or a domain.

In [ ]:
HASH_LENGTHS = {32: "MD5", 40: "SHA-1", 64: "SHA-256"}
DOMAIN_RE = re.compile(r"^(?=.{1,253}$)([a-z0-9]([a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,63}$", re.IGNORECASE)


def detect_indicator_type(indicator: str) -> str:
    """Return 'ipv4', 'ipv6', 'hash' or 'domain'. Raise ValueError otherwise."""
    indicator = indicator.strip()

    try:
        ip = ipaddress.ip_address(indicator)
        return "ipv4" if ip.version == 4 else "ipv6"
    except ValueError:
        pass

    if re.fullmatch(r"[a-fA-F0-9]+", indicator) and len(indicator) in HASH_LENGTHS:
        return "hash"

    if DOMAIN_RE.match(indicator):
        return "domain"

    raise ValueError(f"Unrecognised indicator: {indicator!r}")


for sample in ["8.8.8.8", "google.com", "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f"]:
    print(f"{sample[:40]:<42} -> {detect_indicator_type(sample)}")

## 4. Query VirusTotal

VirusTotal reports how many security engines flagged the indicator in `last_analysis_stats`. The free API allows **4 requests per minute**, so the function waits between calls.

In [ ]:
_last_vt_call = 0.0
VT_MIN_INTERVAL = 15  # seconds -> max 4 requests/minute on the free tier


def query_virustotal(indicator: str, ioc_type: str) -> dict:
    """Query VirusTotal and return a small, clean dictionary."""
    global _last_vt_call

    path = {"ipv4": "ip_addresses", "ipv6": "ip_addresses",
            "domain": "domains", "hash": "files"}[ioc_type]

    wait = VT_MIN_INTERVAL - (time.time() - _last_vt_call)
    if wait > 0 and _last_vt_call:
        time.sleep(wait)

    try:
        resp = requests.get(f"{VT_BASE}/{path}/{indicator}",
                            headers={"x-apikey": VT_API_KEY}, timeout=TIMEOUT)
        _last_vt_call = time.time()
    except requests.RequestException as exc:
        return {"source": "VirusTotal", "ok": False, "error": str(exc)}

    if resp.status_code == 404:
        return {"source": "VirusTotal", "ok": True, "found": False,
                "malicious": 0, "suspicious": 0, "harmless": 0, "undetected": 0}
    if resp.status_code != 200:
        return {"source": "VirusTotal", "ok": False,
                "error": f"HTTP {resp.status_code}: {resp.text[:120]}"}

    attrs = resp.json().get("data", {}).get("attributes", {})
    stats = attrs.get("last_analysis_stats", {})
    return {
        "source": "VirusTotal", "ok": True, "found": True,
        "malicious": stats.get("malicious", 0),
        "suspicious": stats.get("suspicious", 0),
        "harmless": stats.get("harmless", 0),
        "undetected": stats.get("undetected", 0),
        "reputation": attrs.get("reputation"),
        "country": attrs.get("country"),
        "as_owner": attrs.get("as_owner"),
    }

## 5. Query AlienVault OTX

OTX is a community threat-sharing platform. The key number is `pulse_info.count` - how many community "pulses" (threat reports) mention this indicator.

In [ ]:
def query_otx(indicator: str, ioc_type: str) -> dict:
    """Query AlienVault OTX and return a small, clean dictionary."""
    section = {"ipv4": "IPv4", "ipv6": "IPv6", "domain": "domain", "hash": "file"}[ioc_type]

    try:
        resp = requests.get(f"{OTX_BASE}/{section}/{indicator}/general",
                            headers={"X-OTX-API-KEY": OTX_API_KEY}, timeout=TIMEOUT)
    except requests.RequestException as exc:
        return {"source": "AlienVault OTX", "ok": False, "error": str(exc)}

    if resp.status_code == 404:
        return {"source": "AlienVault OTX", "ok": True, "found": False, "pulses": 0, "tags": []}
    if resp.status_code != 200:
        return {"source": "AlienVault OTX", "ok": False,
                "error": f"HTTP {resp.status_code}: {resp.text[:120]}"}

    data = resp.json()
    pulse_info = data.get("pulse_info", {})
    pulses = pulse_info.get("pulses", []) or []

    tags = []
    for p in pulses[:20]:
        tags.extend(p.get("tags", []) or [])
    top_tags = [t for t, _ in pd.Series(tags).value_counts().head(5).items()] if tags else []

    return {
        "source": "AlienVault OTX", "ok": True, "found": True,
        "pulses": pulse_info.get("count", 0),
        "tags": top_tags,
        "country": data.get("country_name"),
    }

## 6. Combine the results into a risk score

The score runs from **0 to 100**:

| Component | Formula | Max |
|---|---|---|
| VirusTotal | `malicious x 4 + suspicious x 2` | 70 |
| AlienVault OTX | number of pulses | 15 |

**Verdict:** below 20 = Low risk, 20-49 = Suspicious, 50 and above = Malicious.

**Design choice:** OTX is capped at 15, so community mentions alone can never push an indicator above "Low risk". Very popular infrastructure such as `8.8.8.8` appears in many pulses without being malicious, so OTX is used to *support* the VirusTotal result, not to decide it alone.

In [ ]:
VT_MAX, OTX_MAX = 70, 15


def compute_risk(vt: dict, otx: dict) -> dict:
    vt_part = min(vt.get("malicious", 0) * 4 + vt.get("suspicious", 0) * 2, VT_MAX) if vt.get("ok") else 0
    otx_part = min(otx.get("pulses", 0), OTX_MAX) if otx.get("ok") else 0
    score = vt_part + otx_part

    if score >= 50:
        verdict = "MALICIOUS"
    elif score >= 20:
        verdict = "SUSPICIOUS"
    else:
        verdict = "LOW RISK"

    return {"vt_points": vt_part, "otx_points": otx_part, "risk_score": score, "verdict": verdict}

## 7. The aggregator

In [ ]:
def analyze(indicator: str) -> dict:
    """Full pipeline for one indicator."""
    indicator = indicator.strip()
    ioc_type = detect_indicator_type(indicator)

    vt = query_virustotal(indicator, ioc_type)
    otx = query_otx(indicator, ioc_type)
    risk = compute_risk(vt, otx)

    return {
        "indicator": indicator,
        "type": ioc_type,
        "checked_at": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC"),
        "virustotal": vt,
        "otx": otx,
        **risk,
    }


def print_report(result: dict) -> None:
    vt, otx = result["virustotal"], result["otx"]
    line = "=" * 62
    print(line)
    print(f"Indicator : {result['indicator']}")
    print(f"Type      : {result['type']}")
    print(f"Checked   : {result['checked_at']}")
    print("-" * 62)

    if vt.get("ok"):
        if vt.get("found"):
            print(f"VirusTotal: {vt['malicious']} malicious, {vt['suspicious']} suspicious, "
                  f"{vt['harmless']} harmless, {vt['undetected']} undetected")
            if vt.get("as_owner"):
                print(f"            Owner: {vt['as_owner']}  |  Country: {vt.get('country')}")
        else:
            print("VirusTotal: not found in database")
    else:
        print(f"VirusTotal: ERROR - {vt.get('error')}")

    if otx.get("ok"):
        if otx.get("found"):
            print(f"OTX       : mentioned in {otx['pulses']} pulse(s)")
            if otx.get("tags"):
                print(f"            Top tags: {', '.join(otx['tags'])}")
        else:
            print("OTX       : not found in database")
    else:
        print(f"OTX       : ERROR - {otx.get('error')}")

    print("-" * 62)
    print(f"Risk score: {result['risk_score']}/100  (VT {result['vt_points']} + OTX {result['otx_points']})")
    print(f"VERDICT   : {result['verdict']}")
    print(line)

## 8. Run it

Try a **known-safe IP**, a **known-safe domain**, and a **known-bad hash**. The hash below is the standard **EICAR test file** - a harmless file that every antivirus engine deliberately flags, so it is the safe way to demonstrate a malicious detection.

You can add your own indicators (for example, from the abuse.ch feeds listed in the README) to the list.

In [ ]:
indicators = [
    "8.8.8.8",                                                            # Google DNS (safe)
    "google.com",                                                         # popular domain (safe)
    "275a021bbfb6489e54d471899f7db9d1663fc695ec2fe2a2c4538aabf651fd0f",  # EICAR test file (flagged on purpose)
]

results = []
for ioc in indicators:
    try:
        result = analyze(ioc)
        print_report(result)
        results.append(result)
    except ValueError as err:
        print(err)

## 9. Save and visualise the results

In [ ]:
rows = [{
    "indicator": r["indicator"], "type": r["type"],
    "vt_malicious": r["virustotal"].get("malicious"),
    "vt_suspicious": r["virustotal"].get("suspicious"),
    "otx_pulses": r["otx"].get("pulses"),
    "risk_score": r["risk_score"], "verdict": r["verdict"],
    "checked_at": r["checked_at"],
} for r in results]

df = pd.DataFrame(rows)
display(df)

with open("results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
df.to_csv("results.csv", index=False)
print("Saved results.json and results.csv")

In [ ]:
colours = {"MALICIOUS": "#c0392b", "SUSPICIOUS": "#e67e22", "LOW RISK": "#2e8b57"}
labels = [(i[:14] + "...") if len(i) > 17 else i for i in df["indicator"]]

plt.figure(figsize=(8, 4))
plt.bar(labels, df["risk_score"], color=[colours[v] for v in df["verdict"]])
plt.axhline(20, linestyle="--", linewidth=1, color="grey")
plt.axhline(50, linestyle="--", linewidth=1, color="grey")
plt.ylim(0, 100)
plt.ylabel("Risk score (0-100)")
plt.title("Threat Intelligence Aggregator - risk score per indicator")
plt.tight_layout()
plt.savefig("risk_scores.png", dpi=150)
plt.show()

## 10. Batch mode (optional)

Analyse many indicators from a text file (one per line). Because of the VirusTotal free-tier limit, this takes about 15 seconds per indicator.

In [ ]:
def analyze_file(path: str) -> list:
    with open(path, encoding="utf-8") as f:
        items = [line.strip() for line in f if line.strip() and not line.startswith("#")]
    out = []
    for item in items:
        try:
            out.append(analyze(item))
        except ValueError as err:
            print(err)
    return out

# Example:  batch = analyze_file("indicators.txt")

## 11. Limitations and future work

- **Free-tier limits:** VirusTotal allows 4 requests/minute, so large batches are slow.
- **Simple scoring:** the weights are hand-chosen and have not been tuned on labelled data. A natural next step is to train a classifier on labelled indicators.
- **Two sources only:** adding AbuseIPDB, URLhaus or Shodan would improve coverage.
- **Point-in-time results:** reputation changes over time, so each result is stamped with the check time.

**Responsible use:** the tool only reads public reputation data. It does not scan or interact with the indicators themselves.